# Pit Stop Submission Blender: Anchor Correction Pipeline

This notebook treats `0.95419` as the new anchor submission. The goal is not to average many files, but to test three controlled correction families around the strongest anchor: micro linear blends, rank blends, and selective row-level corrections.

The output stays intentionally small. Submit candidates are separated by priority: `outputs/max` contains the main files to test first, `outputs/pro` contains secondary diagnostics, and `outputs/report.csv` explains every saved file.


In [ ]:
"""Import libraries, configure paths, and define compact display helpers."""

from pathlib import Path
from html import escape
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except ModuleNotFoundError:
    sns = None
from IPython.display import display, HTML

pd.set_option("display.max_columns", 90)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")
if sns is not None:
    sns.set_theme(style="whitegrid", context="notebook")
else:
    plt.style.use("ggplot")

ID_COL = "id"
TARGET_COL = "PitNextLap"
CLIP_LOW = 1e-7
CLIP_HIGH = 1 - 1e-7

DATASET_CANDIDATES = [
    Path("/kaggle/input/pitstop-blend-inputs"),
    Path("/kaggle/input/pit-stop-blend-inputs"),
    Path("/kaggle/input/blend-dataset"),
    Path("/kaggle/input") / "blend_dataset",
    Path("PredictingPitStop/blend_dataset"),
    Path("blend_dataset"),
]

OUTPUT_ROOT = Path("outputs")
MAX_ROOT = OUTPUT_ROOT / "max"
PRO_ROOT = OUTPUT_ROOT / "pro"
DIAGNOSTIC_ROOT = OUTPUT_ROOT / "diagnostics"
REPORT_PATH = OUTPUT_ROOT / "report.csv"

# Keep output simple and reproducible.
for folder in [MAX_ROOT, PRO_ROOT, DIAGNOSTIC_ROOT]:
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)
if REPORT_PATH.exists():
    REPORT_PATH.unlink()


def find_dataset_dir():
    for path in DATASET_CANDIDATES:
        if (path / "public").exists() and (path / "ours").exists():
            return path
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for path in sorted(kaggle_input.rglob("*")):
            if path.is_dir() and (path / "public").exists() and (path / "ours").exists():
                return path
    raise FileNotFoundError("Could not find a dataset folder containing public/ and ours/ subfolders.")


def show_title(title, subtitle=None):
    subtitle_html = f'<div style="color:#6b7280;font-size:13px;margin-top:4px;">{escape(str(subtitle))}</div>' if subtitle else ""
    display(HTML(
        f"""
        <div style="margin:20px 0 12px 0;padding-bottom:9px;border-bottom:1px solid #e5e7eb;">
          <div style="font-size:20px;font-weight:750;color:#111827;">{escape(str(title))}</div>
          {subtitle_html}
        </div>
        """
    ))


def show_cards(metrics, columns=4):
    width = max(1, int(100 / columns))
    cards = []
    for label, value in metrics.items():
        if isinstance(value, float):
            value = f"{value:.6f}".rstrip("0").rstrip(".")
        elif isinstance(value, int):
            value = f"{value:,}"
        cards.append(
            f"""
            <div style="box-sizing:border-box;width:{width}%;padding:6px;">
              <div style="border:1px solid #e5e7eb;border-radius:8px;padding:12px;background:#fff;">
                <div style="font-size:12px;color:#6b7280;text-transform:uppercase;letter-spacing:.03em;">{escape(str(label))}</div>
                <div style="font-size:21px;font-weight:750;color:#111827;margin-top:5px;">{escape(str(value))}</div>
              </div>
            </div>
            """
        )
    display(HTML(f'<div style="display:flex;flex-wrap:wrap;margin:0 -6px 14px -6px;">{"".join(cards)}</div>'))


def show_table(title, df, max_rows=12, precision=6):
    shown = df.head(max_rows).copy()
    display(HTML(f'<div style="font-size:16px;font-weight:700;color:#111827;margin:14px 0 6px;">{escape(title)}</div>'))
    styler = shown.style.format(precision=precision).set_table_styles([
        {"selector":"th", "props":[("background", "#f9fafb"), ("color", "#374151"), ("font-weight", "700"), ("border-bottom", "1px solid #e5e7eb")]},
        {"selector":"td", "props":[("border-bottom", "1px solid #f3f4f6"), ("font-size", "13px")]},
        {"selector":"table", "props":[("border-collapse", "collapse"), ("width", "100%")]},
    ])
    display(styler)
    if len(df) > max_rows:
        display(HTML(f'<div style="color:#6b7280;font-size:12px;margin-top:4px;">Showing {max_rows} of {len(df)} rows.</div>'))


def corr(a, b):
    return float(np.corrcoef(np.asarray(a), np.asarray(b))[0, 1])


def mean_abs_delta(a, b):
    return float(np.abs(np.asarray(a) - np.asarray(b)).mean())


def clip_pred(pred):
    return np.clip(np.asarray(pred, dtype=float), CLIP_LOW, CLIP_HIGH)


## 1. Load Blend Dataset

The loader reads the structured dataset only. The `public/super` folder contains the current strongest anchor files, while older public and internal submissions are kept as support signals for diagnostics and conservative corrections.


In [ ]:
"""Load all valid submission files from the structured blend dataset."""

dataset_dir = find_dataset_dir()
predictions = {}
records = []
base_ids = None


def load_submission(path, source):
    global base_ids
    df = pd.read_csv(path)
    if ID_COL not in df.columns:
        return None
    target_candidates = [col for col in df.columns if col != ID_COL]
    if TARGET_COL in df.columns:
        target_col = TARGET_COL
    elif len(target_candidates) == 1:
        target_col = target_candidates[0]
    else:
        return None

    df = df[[ID_COL, target_col]].rename(columns={target_col: TARGET_COL})
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    if df[TARGET_COL].isna().any() or df[ID_COL].duplicated().any():
        return None

    if base_ids is None:
        base_ids = df[ID_COL].copy()
    elif not base_ids.equals(df[ID_COL]):
        return None

    group = path.parent.name
    safe_stem = path.stem.replace(".", "_").replace("-", "_")
    name = f"{source}_{group}_{safe_stem}"
    pred = clip_pred(df[TARGET_COL].to_numpy(dtype=float))
    predictions[name] = pred

    public_score = np.nan
    if source == "public":
        try:
            public_score = float(path.stem.split("_")[0])
        except ValueError:
            public_score = np.nan

    return {
        "name": name,
        "source": source,
        "group": group,
        "public_score": public_score,
        "rows": len(df),
        "mean": pred.mean(),
        "std": pred.std(),
        "file": str(path.relative_to(dataset_dir)),
    }


for source in ["public", "ours"]:
    folder = dataset_dir / source
    if folder.exists():
        for path in sorted(folder.rglob("*.csv")):
            row = load_submission(path, source)
            if row is not None:
                records.append(row)

if not records:
    raise FileNotFoundError("No valid submissions were loaded from the structured dataset.")

input_summary = pd.DataFrame(records).sort_values(
    ["source", "group", "public_score", "name"], ascending=[True, True, False, True]
).reset_index(drop=True)
input_summary.to_csv(DIAGNOSTIC_ROOT / "inputs.csv", index=False)

show_title("Loaded blend dataset", f"Dataset folder: {dataset_dir}")
show_cards({
    "input files": len(input_summary),
    "public files": int(input_summary["source"].eq("public").sum()),
    "own files": int(input_summary["source"].eq("ours").sum()),
    "rows": len(base_ids),
})
show_table(
    "Structured inputs",
    input_summary[["name", "source", "group", "public_score", "mean", "std", "file"]],
    max_rows=24,
)


## 2. Anchor Diagnostics

Before blending, we check whether `0.95419` is a useful new source or just a duplicate of the previous `0.95418` anchor. The goal is to find close but non-identical support signals: enough diversity to correct mistakes, but not so much that the anchor gets diluted.


In [ ]:
"""Select the anchor and compare the strongest public submissions around it."""

super_meta = input_summary[input_summary["group"].eq("super")].sort_values("public_score", ascending=False).copy()
if super_meta.empty:
    raise RuntimeError("No public/super submissions were found.")

anchor_names = super_meta[np.isclose(super_meta["public_score"], 0.95419, atol=1e-8)]["name"].tolist()
if not anchor_names:
    raise RuntimeError("The 0.95419 submission is required in public/super.")
s19_name = anchor_names[0]
s19_pred = predictions[s19_name]

s18_names = super_meta[np.isclose(super_meta["public_score"], 0.95418, atol=1e-8)]["name"].tolist()
if not s18_names:
    raise RuntimeError("The 0.95418 support submission is required in public/super.")
s18_name = s18_names[0]
s18_pred = predictions[s18_name]

s11_names = super_meta[np.isclose(super_meta["public_score"], 0.95411, atol=1e-8)]["name"].tolist()
if not s11_names:
    raise RuntimeError("At least one 0.95411 support submission is required in public/super.")
s11_pred = clip_pred(np.vstack([predictions[name] for name in s11_names]).mean(axis=0))

super_names = super_meta["name"].tolist()
super_matrix = pd.DataFrame({name: predictions[name] for name in super_names})
super_corr = super_matrix.corr()

anchor_rows = []
for name in super_names:
    pred = predictions[name]
    anchor_rows.append({
        "signal": name,
        "public_score": float(super_meta.loc[super_meta["name"].eq(name), "public_score"].iloc[0]),
        "mean": pred.mean(),
        "std": pred.std(),
        "corr_to_s19": corr(pred, s19_pred),
        "delta_to_s19": mean_abs_delta(pred, s19_pred),
    })
anchor_summary = pd.DataFrame(anchor_rows).sort_values("public_score", ascending=False).reset_index(drop=True)
anchor_summary.to_csv(DIAGNOSTIC_ROOT / "anchor_diagnostics.csv", index=False)

show_title("Anchor diagnostics", "0.95419 becomes the base; all other signals are measured against it")
show_cards({
    "anchor": "0.95419",
    "support 0.95418 delta": mean_abs_delta(s18_pred, s19_pred),
    "support 0.95418 corr": corr(s18_pred, s19_pred),
    "super files": len(super_names),
})
show_table("Super signals", anchor_summary, max_rows=len(anchor_summary))

fig, ax = plt.subplots(figsize=(7, 5))
if sns is not None:
    sns.heatmap(super_corr, cmap="viridis", annot=True, fmt=".5f", cbar=False, ax=ax)
else:
    im = ax.imshow(super_corr.values, cmap="viridis")
    ax.set_xticks(range(len(super_corr.columns)), super_corr.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(super_corr.index)), super_corr.index)
    fig.colorbar(im, ax=ax)
ax.set_title("Super submission correlation")
plt.tight_layout()
plt.show()


## 3. Build Support Signals

The old blender is now used only as an in-memory reference. `tb` represents the previous `0.95410` area, while `s18` is the closest strong support to `s19`. These signals help us test whether the anchor has correctable local errors.


In [ ]:
"""Build support signals used by the correction methods without saving them as submissions."""


def names_by_group(group):
    return input_summary[input_summary["group"].eq(group)]["name"].tolist()


def mean_predictions(names):
    if not names:
        raise RuntimeError("Cannot average an empty list of predictions.")
    return clip_pred(np.vstack([predictions[name] for name in names]).mean(axis=0))


public_core = input_summary[input_summary["group"].eq("core")].sort_values("public_score", ascending=False)["name"].head(6).tolist()
public_diverse = names_by_group("diverse")
top_external = names_by_group("top_external")

if not public_core:
    raise RuntimeError("public/core is empty; cannot build b10 reference.")
if not public_diverse:
    raise RuntimeError("public/diverse is empty; cannot build b10 reference.")
if not top_external:
    raise RuntimeError("public/top_external is empty; cannot build tx/tb reference.")

core_pred = mean_predictions(public_core)
diverse_pred = predictions[public_diverse[0]]
b10_pred = clip_pred(0.950 * core_pred + 0.050 * diverse_pred)
tx_pred = mean_predictions(top_external)
tb_pred = clip_pred(0.900 * tx_pred + 0.100 * b10_pred)

support_signals = {
    "s19": s19_pred,
    "s18": s18_pred,
    "s11": s11_pred,
    "tb": tb_pred,
    "tx": tx_pred,
    "b10": b10_pred,
}

support_rows = []
for signal, pred in support_signals.items():
    support_rows.append({
        "signal": signal,
        "role": {
            "s19": "new anchor",
            "s18": "closest strong support",
            "s11": "0.95411 support mean",
            "tb": "previous best blend area",
            "tx": "older 0.95409 pair mean",
            "b10": "old public-core baseline",
        }[signal],
        "mean": pred.mean(),
        "std": pred.std(),
        "corr_to_s19": corr(pred, s19_pred),
        "delta_to_s19": mean_abs_delta(pred, s19_pred),
    })
support_summary = pd.DataFrame(support_rows).sort_values("delta_to_s19").reset_index(drop=True)
support_summary.to_csv(DIAGNOSTIC_ROOT / "support_signals.csv", index=False)

show_title("Support signals", "Only s18 is close enough for primary corrections; tb/s11 are secondary context")
show_table("Support summary", support_summary, max_rows=len(support_summary))


## 4. Method 1: Linear Micro Blends

Linear micro blends test whether a tiny amount of the closest support submission improves calibration. The weights are deliberately small because `s19` is already the strongest known file.


In [ ]:
"""Create linear micro-blend candidates around the 0.95419 anchor."""

linear_candidates = {
    "l99": {
        "pred": 0.990 * s19_pred + 0.010 * s18_pred,
        "formula": "0.990*s19 + 0.010*s18",
        "tier": "pro",
        "priority": 5,
        "reason": "Very small calibration test from the closest support.",
    },
    "l975": {
        "pred": 0.975 * s19_pred + 0.025 * s18_pred,
        "formula": "0.975*s19 + 0.025*s18",
        "tier": "max",
        "priority": 2,
        "reason": "Main linear micro-blend candidate.",
    },
    "l95": {
        "pred": 0.950 * s19_pred + 0.050 * s18_pred,
        "formula": "0.950*s19 + 0.050*s18",
        "tier": "pro",
        "priority": 6,
        "reason": "Stronger linear correction; useful only if small blends help.",
    },
}

linear_summary = pd.DataFrame([
    {
        "candidate": key,
        "tier": spec["tier"],
        "formula": spec["formula"],
        "corr_to_s19": corr(spec["pred"], s19_pred),
        "delta_to_s19": mean_abs_delta(spec["pred"], s19_pred),
        "reason": spec["reason"],
    }
    for key, spec in linear_candidates.items()
]).sort_values("delta_to_s19")
linear_summary.to_csv(DIAGNOSTIC_ROOT / "linear_candidates.csv", index=False)

show_title("Linear micro blends", "Small global corrections from s18")
show_table("Linear candidates", linear_summary, max_rows=len(linear_summary))


## 5. Method 2: Rank Blends

Rank blends focus on ordering instead of raw probability values. Since the leaderboard metric is AUC, a candidate can improve even if its probability calibration is almost unchanged. The final probability distribution is mapped back to the `s19` distribution.


In [ ]:
"""Create rank-blend candidates that preserve the anchor probability distribution."""


def normalized_rank(values):
    order = np.argsort(values, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.linspace(0.0, 1.0, len(values))
    return ranks


def rank_blend(anchor, support, support_weight):
    anchor_rank = normalized_rank(anchor)
    support_rank = normalized_rank(support)
    blended_rank = (1 - support_weight) * anchor_rank + support_weight * support_rank
    order = np.argsort(blended_rank, kind="mergesort")
    sorted_anchor_values = np.sort(anchor)
    out = np.empty_like(anchor, dtype=float)
    out[order] = sorted_anchor_values
    return clip_pred(out)


rank_candidates = {
    "r975": {
        "pred": rank_blend(s19_pred, s18_pred, 0.025),
        "formula": "rank 0.975*s19 + 0.025*s18",
        "tier": "max",
        "priority": 3,
        "reason": "Main rank-order test with tiny s18 influence.",
    },
    "r95": {
        "pred": rank_blend(s19_pred, s18_pred, 0.050),
        "formula": "rank 0.950*s19 + 0.050*s18",
        "tier": "pro",
        "priority": 7,
        "reason": "Stronger rank-order correction.",
    },
}

rank_summary = pd.DataFrame([
    {
        "candidate": key,
        "tier": spec["tier"],
        "formula": spec["formula"],
        "corr_to_s19": corr(spec["pred"], s19_pred),
        "delta_to_s19": mean_abs_delta(spec["pred"], s19_pred),
        "reason": spec["reason"],
    }
    for key, spec in rank_candidates.items()
]).sort_values("delta_to_s19")
rank_summary.to_csv(DIAGNOSTIC_ROOT / "rank_candidates.csv", index=False)

show_title("Rank blends", "Ranking corrections with the original s19 probability distribution")
show_table("Rank candidates", rank_summary, max_rows=len(rank_summary))


## 6. Method 3: Selective Correction

Selective corrections keep almost all rows unchanged and only move rows where the anchor and support disagree the most. This is useful when a strong anchor is already close to optimum and a global blend would damage too many correct rows.


In [ ]:
"""Create selective row-level correction candidates on the largest s19/s18 disagreements."""


def selective_blend(anchor, support, fraction, support_weight, direction="both"):
    anchor = np.asarray(anchor, dtype=float)
    support = np.asarray(support, dtype=float)
    delta = support - anchor
    if direction == "up":
        eligible = delta > 0
    elif direction == "down":
        eligible = delta < 0
    else:
        eligible = np.ones_like(delta, dtype=bool)

    scores = np.where(eligible, np.abs(delta), -np.inf)
    n_select = max(1, int(round(len(anchor) * fraction)))
    selected = np.argpartition(scores, -n_select)[-n_select:]
    selected = selected[np.isfinite(scores[selected])]

    out = anchor.copy()
    out[selected] = (1 - support_weight) * anchor[selected] + support_weight * support[selected]
    return clip_pred(out), len(selected)


c02_pred, c02_rows = selective_blend(s19_pred, s18_pred, fraction=0.020, support_weight=0.100, direction="both")
up02_pred, up02_rows = selective_blend(s19_pred, s18_pred, fraction=0.020, support_weight=0.100, direction="up")
dn02_pred, dn02_rows = selective_blend(s19_pred, s18_pred, fraction=0.020, support_weight=0.100, direction="down")

selective_candidates = {
    "c02": {
        "pred": c02_pred,
        "formula": "top 2% |s18-s19|: 0.900*s19 + 0.100*s18",
        "tier": "max",
        "priority": 4,
        "changed_rows": c02_rows,
        "reason": "Main selective correction on the most disputed rows.",
    },
    "up02": {
        "pred": up02_pred,
        "formula": "top 2% where s18>s19: 0.900*s19 + 0.100*s18",
        "tier": "pro",
        "priority": 8,
        "changed_rows": up02_rows,
        "reason": "Directional test for possible underestimation by s19.",
    },
    "dn02": {
        "pred": dn02_pred,
        "formula": "top 2% where s18<s19: 0.900*s19 + 0.100*s18",
        "tier": "pro",
        "priority": 9,
        "changed_rows": dn02_rows,
        "reason": "Directional test for possible overestimation by s19.",
    },
}

selective_summary = pd.DataFrame([
    {
        "candidate": key,
        "tier": spec["tier"],
        "formula": spec["formula"],
        "changed_rows": spec["changed_rows"],
        "corr_to_s19": corr(spec["pred"], s19_pred),
        "delta_to_s19": mean_abs_delta(spec["pred"], s19_pred),
        "reason": spec["reason"],
    }
    for key, spec in selective_candidates.items()
]).sort_values("delta_to_s19")
selective_summary.to_csv(DIAGNOSTIC_ROOT / "selective_candidates.csv", index=False)

show_title("Selective corrections", "Only the largest anchor/support disagreements are changed")
show_table("Selective candidates", selective_summary, max_rows=len(selective_summary))


## 7. Save Final Outputs

Only a compact set is saved. `max` contains the main submit order, while `pro` contains secondary checks that should be used only if there are enough attempts or if a method family shows promise.


In [ ]:
"""Save selected candidates to outputs/max and outputs/pro, then build one report file."""


def save_submission(path, pred):
    pred = clip_pred(pred)
    pd.DataFrame({ID_COL: base_ids.values, TARGET_COL: pred}).to_csv(path, index=False)

all_candidates = {
    "s19": {
        "pred": s19_pred,
        "method": "anchor",
        "tier": "max",
        "priority": 1,
        "formula": "0.95419 raw anchor",
        "changed_rows": 0,
        "reason": "Verify the new strongest anchor unchanged.",
    },
}
for key, spec in linear_candidates.items():
    all_candidates[key] = {"method": "linear", "changed_rows": len(s19_pred), **spec}
for key, spec in rank_candidates.items():
    all_candidates[key] = {"method": "rank", "changed_rows": len(s19_pred), **spec}
for key, spec in selective_candidates.items():
    all_candidates[key] = {"method": "selective", **spec}

rows = []
for key, spec in sorted(all_candidates.items(), key=lambda item: item[1]["priority"]):
    pred = clip_pred(spec["pred"])
    folder = MAX_ROOT if spec["tier"] == "max" else PRO_ROOT
    file_path = folder / f"{key}.csv"
    save_submission(file_path, pred)
    rows.append({
        "priority": spec["priority"],
        "tier": spec["tier"],
        "candidate": key,
        "method": spec["method"],
        "file": str(file_path.relative_to(OUTPUT_ROOT)),
        "formula": spec["formula"],
        "mean": pred.mean(),
        "std": pred.std(),
        "corr_to_s19": corr(pred, s19_pred),
        "delta_to_s19": mean_abs_delta(pred, s19_pred),
        "changed_rows": spec["changed_rows"],
        "reason": spec["reason"],
    })

report = pd.DataFrame(rows).sort_values("priority").reset_index(drop=True)
report.to_csv(REPORT_PATH, index=False)

show_title("Saved output files", "Upload max first; use pro for method diagnostics")
show_cards({
    "max files": int(report["tier"].eq("max").sum()),
    "pro files": int(report["tier"].eq("pro").sum()),
    "total files": len(report),
    "report": "outputs/report.csv",
})
show_table(
    "Submit order",
    report[["priority", "tier", "candidate", "method", "file", "formula", "delta_to_s19", "changed_rows", "reason"]],
    max_rows=len(report),
)


## 8. Candidate Movement

The final plots are meant to make the output easy to read. A good candidate should usually stay very close to `s19`, unless the method intentionally tests a larger correction. If a method improves the leaderboard, the next iteration should search narrowly around that method only.


In [ ]:
"""Visualize candidate movement and correction shape relative to the anchor."""

plot_df = report[report["candidate"].ne("s19")].copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
if sns is not None:
    sns.barplot(data=plot_df, y="candidate", x="delta_to_s19", hue="method", dodge=False, ax=axes[0])
    sns.barplot(data=plot_df, y="candidate", x="changed_rows", hue="method", dodge=False, ax=axes[1])
    for ax in axes:
        legend = ax.get_legend()
        if legend is not None:
            legend.remove()
else:
    axes[0].barh(plot_df["candidate"], plot_df["delta_to_s19"])
    axes[1].barh(plot_df["candidate"], plot_df["changed_rows"])
axes[0].set_title("Movement from s19")
axes[0].set_xlabel("mean absolute delta")
axes[1].set_title("Rows changed")
axes[1].set_xlabel("rows")
for ax in axes:
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7.5, 4.3))
for candidate in ["l975", "r975", "c02"]:
    row = report[report["candidate"].eq(candidate)].iloc[0]
    pred = pd.read_csv(OUTPUT_ROOT / row["file"])[TARGET_COL].to_numpy(dtype=float)
    ax.hist(pred - s19_pred, bins=80, alpha=0.45, label=candidate)
ax.set_title("Main candidate corrections relative to s19")
ax.set_xlabel("prediction delta")
ax.set_ylabel("rows")
ax.legend()
plt.tight_layout()
plt.show()

show_title("Final recommendation", "The first four files answer four different hypotheses")
show_table("Max candidates", report[report["tier"].eq("max")][["priority", "candidate", "method", "file", "reason"]], max_rows=10)
show_table("Pro candidates", report[report["tier"].eq("pro")][["priority", "candidate", "method", "file", "reason"]], max_rows=10)


## Final Notes

This blender is designed as a controlled experiment around `s19`. The first submissions should be `s19`, `l975`, `r975`, and `c02`. Their leaderboard results tell us which direction deserves the next narrow search: linear calibration, rank correction, or selective row-level correction.

If none of these improves the score, the current `0.95419` file is probably already a strong local optimum for this set of sources, and the next improvement will likely require another genuinely different high-scoring external submission.
